# 06 — Regulators (TF–TF network and master regulators per state)

Cancer-agnostic port of GBM `06_regulators.ipynb`. **Only the config cell below needs editing for a new cancer.**
Needs `filteredCausalResults.csv` (causal filter at the end of notebook 02) and the MINER outputs in `<cancer>_output/`.

Causal flows are first limited to published driver genes of this cancer (`driver_tiers.py`, tiers known / candidate; cached in `{prefix}_data/driver_genes.csv`), then TFs that are barely expressed are dropped. Extra outputs: `driverCausalResults.csv` (flows kept, with `Tier`), `tf_expression_filter.csv`.

> **GBM note.** The TF expression filter uses the RNA-seq patients only (`gbm_log2tpm.csv`); the network itself uses all patients (`gbm_zscored.csv`).

In [1]:
# Necessary imports
import os
import pickle
import time
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import miner.miner as miner
from miner.miner import *

# Repository root: holds config.py (paths and per-cancer settings) and driver_tiers.py
import sys
repo_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)
import config

# CANCER CONFIGURATION: the cancer and the data root come from config.py
CANCER     = config.CANCER

# folder holding cancer run of MINER
base_dir   = config.BASE_DIR

# this cancer's own folder: holds <prefix>_data, <prefix>_notebooks and <prefix>_output
cancer_dir = os.path.join(base_dir, CANCER.lower())

# this cancer's expression / mutation files (written by 01 and 02)
data_dir   = os.path.join(cancer_dir, f"{CANCER.lower()}_data")

# corresponding MINER outputs (regulons, eigengenes, ...)
output_dir = os.path.join(cancer_dir, f"{CANCER.lower()}_output")

# z-scored expression data, gene symbols x patients
expr_path  = os.path.join(data_dir, f"{CANCER.lower()}_zscored.csv")

# TF expression filter: a TF must reach MIN_TPM in at least MIN_FRAC of tumours to be in the TF-TF network
MIN_TPM  = 1.0
MIN_FRAC = 0.20
log2_path = os.path.join(data_dir, f"{CANCER.lower()}_log2tpm.csv")

# Driver gene tiers: published driver calls per cancer (module driver_tiers.py in the repository root, shared by 02, 06 and 07)
from driver_tiers import get_driver_tiers

assert os.path.isdir(output_dir), f"missing output folder: {output_dir}"
assert os.path.exists(expr_path), f"missing expression file: {expr_path}"
miner_dir = os.path.dirname(miner.__file__)
print(f"Cancer: {CANCER} | outputs: {output_dir}")

Cancer: GBM | outputs: data\gbm\gbm_output


In [2]:
# causal_results: filtered somatic mutation causal flows (mutation, TF, regulon)
# Only the three columns this notebook uses are read, as categories: in heavily mutated cancers this file has millions of
# rows (UCEC: 2.7 million, 400 MB) and loading all 13 columns can exhaust memory and kill the kernel
causal_results = pd.read_csv(os.path.join(output_dir, "filteredCausalResults.csv"),
                             usecols=["Mutation", "Regulator", "Regulon"], dtype="category")
causal_results['Regulon'] = causal_results['Regulon'].astype(str).str.replace('R-', '', regex=False)  # match eigengene index

# exp_data: z-scored expression, gene symbols x patients
exp_data = pd.read_csv(expr_path, index_col=0)

# eigengenes: PC1 score per regulon per patient
eigengenes = pd.read_csv(os.path.join(output_dir, "eigengenes.csv"), index_col=0)

# transcriptional_states: {state ID: [patient IDs]}
transcriptional_states = read_json(os.path.join(output_dir, "transcriptional_states.json"))

# Ensembl <-> gene symbol mappings
idmap = pd.read_csv(os.path.join(miner_dir, "data", "identifier_mappings.txt"), sep="\t")
gene_name_map     = idmap[idmap['Source'] == 'Gene Name'][['Preferred_Name', 'Name']]
ensembl_to_symbol = dict(zip(gene_name_map['Preferred_Name'], gene_name_map['Name']))
symbol_to_ensembl = {v: k for k, v in ensembl_to_symbol.items()}

# TFBSDB v1 + v2 (computational) + ChEA (experimental) binding site databases
nd = os.path.join(miner_dir, "data", "network_dictionaries")
tfbsdb_tf_to_genes  = pickle.load(open(os.path.join(nd, "tfbsdb_tf_to_genes.pkl"), 'rb'))
tfbsdb2_tf_to_genes = pickle.load(open(os.path.join(nd, "tfbsdb2_tf_to_genes.pkl"), 'rb'))
chea_seq            = pickle.load(open(os.path.join(nd, "chea_seq.pkl"), 'rb'))

print("causal_results:", causal_results.shape)
print("exp_data:", exp_data.shape)
print("eigengenes:", eigengenes.shape)
print("transcriptional_states:", len(transcriptional_states), "states")

causal_results: (26988, 3)
exp_data: (9790, 550)
eigengenes: (3961, 550)
transcriptional_states: 24 states


In [3]:
# Driver filter: keep only causal flows that start from a published driver gene of this cancer
# In heavily mutated cancers most frequently mutated genes are passengers (TTN, MUC16, CSMD3, ...) that track total
# mutation burden, so their "causal" flows are confounded. Same gene list as 02 and 07 (driver_tiers.py):
# both tiers need an IntOGen driver call for this cancer: "known" = also in the Cancer Gene Census, "candidate" = not in it
driver_table = get_driver_tiers(CANCER, data_dir)
tier_of = dict(zip(driver_table["Gene"], driver_table["Tier"]))

if len(driver_table) == 0:
    # back to plain text columns
    causal_results = causal_results.astype(str)
    causal_results["Tier"] = "unfiltered"
    print(f"No driver list for {CANCER}: all {causal_results['Mutation'].nunique()} mutated genes are kept")
else:
    n_flows, n_genes, n_tfs = len(causal_results), causal_results["Mutation"].nunique(), causal_results["Regulator"].nunique()
    # back to plain text columns now that the table is small
    # Pathway mutations (names from the pathway database, e.g. PID_...) are kept as their own tier
    is_pathway = causal_results["Mutation"].astype(str).str.startswith(config.PATHWAY_PREFIX)
    causal_results = causal_results[causal_results["Mutation"].isin(tier_of.keys()) | is_pathway].astype(str)
    causal_results["Tier"] = causal_results["Mutation"].map(tier_of).fillna("pathway")
    print(f"Causal flows: {n_flows} -> {len(causal_results)} | mutated genes: {n_genes} -> {causal_results['Mutation'].nunique()} | "
          f"TFs: {n_tfs} -> {causal_results['Regulator'].nunique()}")
    print("Flows per driver:")
    print(causal_results.groupby(["Tier", "Mutation"]).size().sort_values(ascending=False).to_string())

    # Listed drivers without flows: mutated in < 5% of patients, or mainly altered by copy number / fusion
    print("Listed drivers with no flows:", len(set(tier_of) - set(causal_results["Mutation"])), "of", len(tier_of))

# Saved so 06's input after the filter can be inspected
causal_results.to_csv(os.path.join(output_dir, "driverCausalResults.csv"), index=False)


Driver genes for GBM loaded from cache: {'known': 45, 'candidate': 3}
IntOGen score >= 0.6: 12 of 48 driver calls kept: {'known': 12}
Causal flows: 26988 -> 26231 | mutated genes: 205 -> 181 | TFs: 294 -> 294
Flows per driver:
Tier     Mutation                             
known    ATRX                                     1042
         IDH1                                     1022
pathway  PID_MYC_ACTIV_PATHWAY                     822
         PID_BARD1_PATHWAY                         799
known    TP53                                      784
pathway  PID_P53_REGULATION_PATHWAY                774
         PID_HDAC_CLASSIII_PATHWAY                 759
         PID_HIF1A_PATHWAY                         750
         PID_AURORA_A_PATHWAY                      735
         PID_LKB1_PATHWAY                          711
         PID_AP1_PATHWAY                           580
         PID_P38_ALPHA_BETA_DOWNSTREAM_PATHWAY     561
         PID_ILK_PATHWAY                           537
         PI

In [4]:
# TFs implicated in causal flows
causal_tfs_symbols = causal_results['Regulator'].unique().tolist()

# Drop TFs that are barely expressed in this cancer
# Without this, TFs with near-zero noisy expression get LASSO edges and, having no regulators of their own, top the out/in ranking
# log2(TPM+1), gene symbols x patients (from 01); only the TF rows are kept in memory
log_tpm = pd.read_csv(log2_path, index_col=0)
tf_log_tpm = log_tpm.reindex(causal_tfs_symbols).dropna()
del log_tpm

# Fraction of tumours in which each TF reaches MIN_TPM
frac_expressed = (tf_log_tpm >= np.log2(MIN_TPM + 1)).mean(axis=1)
expressed_tfs  = frac_expressed[frac_expressed >= MIN_FRAC].index.tolist()
dropped_tfs    = sorted(set(causal_tfs_symbols) - set(expressed_tfs))

print(f"TFs kept: {len(expressed_tfs)} / {len(causal_tfs_symbols)} (>= {MIN_TPM} TPM in >= {MIN_FRAC:.0%} of tumours)")
print("Dropped:", dropped_tfs)

# Saved so the choice of threshold can be reviewed
frac_expressed.rename("frac_expressed").to_csv(os.path.join(output_dir, "tf_expression_filter.csv"))

causal_tfs_symbols = expressed_tfs
causal_tfs_ensembl = [t for t in (symbol_to_ensembl.get(tf) for tf in causal_tfs_symbols) if t is not None]
print(f"TFs from causal results: {len(causal_tfs_symbols)} | mapped to Ensembl: {len(causal_tfs_ensembl)}")

# TF (Ensembl) -> genes it binds in any of the three databases
combined_tf_to_genes = {}
for tf in causal_tfs_ensembl:
    genes = set()
    for db in (tfbsdb_tf_to_genes, tfbsdb2_tf_to_genes, chea_seq):
        if tf in db:
            genes.update(db[tf])
    combined_tf_to_genes[tf] = list(genes)

# Expression of the causal TFs (TFs x patients); TFs without expression data are dropped
tf_exp = exp_data.reindex(causal_tfs_symbols).dropna()
print(f"TFs with expression data: {tf_exp.shape[0]}")

TFs kept: 279 / 294 (>= 1.0 TPM in >= 20% of tumours)
Dropped: ['CDX1', 'CEBPE', 'ELF5', 'GATA6', 'GFI1', 'HOXB5', 'HOXB6', 'IKZF3', 'IRF4', 'KLF1', 'LHX3', 'PITX2', 'POU2AF1', 'SOX30', 'TFAP2B']
TFs from causal results: 279 | mapped to Ensembl: 279
TFs with expression data: 279


In [5]:
# TF-TF network with LASSO: A -> B if A has a binding site in B's promoter AND A's expression predicts B's (non-zero coef)
start = time.time()
tf_tf_edges = []
print(f"Building TF-TF network for {tf_exp.shape[0]} TFs...")

for i, target_tf in enumerate(tf_exp.index):
    if i % 25 == 0:
        print(f"{i}/{tf_exp.shape[0]}")

    # Each TF in turn is the target whose expression is to be predicted
    target_ensembl = symbol_to_ensembl.get(target_tf)
    if target_ensembl is None:
        continue

    # Candidate regulators: causal TFs with a binding site in the target's promoter (needs at least 2)
    predictor_tfs = []
    for source_ensembl, target_genes in combined_tf_to_genes.items():
        if target_ensembl in target_genes:
            source_symbol = ensembl_to_symbol.get(source_ensembl)
            if source_symbol and source_symbol in tf_exp.index and source_symbol != target_tf:
                predictor_tfs.append(source_symbol)
    if len(predictor_tfs) < 2:
        continue

    # LASSO: predict the target's expression from its candidate regulators
    # Non-zero coefficients become edges; the sign gives activation (1) or repression (-1)
    X_scaled = StandardScaler().fit_transform(tf_exp.loc[predictor_tfs].T.values)
    y = tf_exp.loc[target_tf].values
    try:
        lasso = LassoCV(cv=5, max_iter=10000, random_state=42).fit(X_scaled, y)
        for j, coef in enumerate(lasso.coef_):
            if coef != 0:
                tf_tf_edges.append({'Source': predictor_tfs[j], 'Target': target_tf,
                                    'Edge': 1 if coef > 0 else -1, 'Coef': coef})
    except Exception:
        continue

tf_network = pd.DataFrame(tf_tf_edges)
print(f"Completed in {(time.time()-start)/60:.1f} minutes")
print(f"TF-TF network edges: {len(tf_network)} (GBM: 11848)")
print(f"Unique source TFs: {tf_network['Source'].nunique()} | unique target TFs: {tf_network['Target'].nunique()}")
tf_network.to_csv(os.path.join(output_dir, "tf_tf_network.csv"), index=False)

Building TF-TF network for 279 TFs...
0/279
25/279
50/279
75/279
100/279
125/279
150/279
175/279
200/279
225/279
250/279
275/279
Completed in 0.9 minutes
TF-TF network edges: 14023 (GBM: 11848)
Unique source TFs: 279 | unique target TFs: 279


In [6]:
# out/in degree ratio: high = master regulator (regulates many, regulated by few)
outdegree = tf_network.groupby('Source')['Target'].count().rename('outdegree')
indegree  = tf_network.groupby('Target')['Source'].count().rename('indegree')
network_degrees = pd.DataFrame({'outdegree': outdegree, 'indegree': indegree}).fillna(0)
network_degrees['out/in'] = network_degrees['outdegree'] / (network_degrees['indegree'] + 1)
network_degrees['Alt_ID'] = network_degrees.index       # miner source expects this column
network_degrees = network_degrees.sort_values('out/in', ascending=False)
print("Top 10 TFs by outdegree/indegree ratio:")
print(network_degrees.head(10))
network_degrees.to_csv(os.path.join(output_dir, "tf_network_degrees.csv"))

# Coverage dictionary: TF -> TFs it regulates
coverage_dict = {tf: tf_network.loc[tf_network['Source'] == tf, 'Target'].tolist() for tf in tf_network['Source'].unique()}

Top 10 TFs by outdegree/indegree ratio:
        outdegree  indegree    out/in  Alt_ID
ZNF35         122        12  9.384615   ZNF35
ZNF263         89        16  5.235294  ZNF263
SP4           114        21  5.181818     SP4
MSX2           74        17  4.111111    MSX2
HOXC13         65        15  4.062500  HOXC13
TFAP2A        127        33  3.735294  TFAP2A
IRF3          108        28  3.724138    IRF3
FOXC2          96        26  3.555556   FOXC2
MAZ            93        26  3.444444     MAZ
ELF4          107        31  3.343750    ELF4


In [7]:
# Master regulators per state (same procedure as GBM, no tolerance cutoff):
# 1. regulons differentially active in the state vs the rest (Welch t-test on eigengenes)
# 2. causal TFs that regulate those regulons
# 3. state-specific TF-TF subnetwork -> unmix + select_optimal_tfs -> rank by out/in
# States with <3 patients are skipped automatically
state_master_regulators_all = {}

for state_id, state_patients in transcriptional_states.items():
    # Patients in this state vs everyone else
    rest_patients    = list(set(eigengenes.columns) - set(state_patients))
    state_pts_in_eig = [p for p in state_patients if p in eigengenes.columns]
    if len(state_pts_in_eig) < 3 or len(rest_patients) < 3:
        continue

    # 1. Regulons differentially active in this state (p <= 0.05)
    _, pvals = scipy_stats.ttest_ind(eigengenes[state_pts_in_eig].values, eigengenes[rest_patients].values,
                                     axis=1, equal_var=False)
    state_regulons = eigengenes.index[pvals <= 0.05].astype(str).tolist()

    # 2. Causal TFs that regulate those regulons and are sources in the TF-TF network
    state_tfs = causal_results[causal_results['Regulon'].isin(state_regulons)]['Regulator'].unique().tolist()
    state_tfs = [tf for tf in state_tfs if tf in coverage_dict]
    if len(state_tfs) < 2:
        continue

    # 3. TF-TF subnetwork restricted to this state's TFs, with its own in/out degrees
    state_out = {tf: [t for t in coverage_dict[tf] if t in state_tfs] for tf in state_tfs}
    state_degrees = pd.DataFrame({
        'outdegree': [len(state_out[tf]) for tf in state_tfs],
        'indegree':  [sum(1 for targets in state_out.values() if tf in targets) for tf in state_tfs],
        'Alt_ID':    state_tfs
    }, index=state_tfs)
    state_degrees['out/in'] = state_degrees['outdegree'] / (state_degrees['indegree'] + 1)

    # Adjacency matrix of the subnetwork (row TF regulates column TF)
    coin_matrix = pd.DataFrame(0, index=state_tfs, columns=state_tfs)
    for tf, targets in state_out.items():
        for t in targets:
            coin_matrix.loc[tf, t] = 1
    if coin_matrix.sum().sum() == 0:
        continue

    # miner's unmix() can emit empty clusters once the remaining TFs have no edges between them;
    # select_optimal_tfs() then fails with 'argmax of an empty sequence'. Edge-less TFs can't be master regulators, so drop them.
    unmix_tst        = [cl for cl in unmix(coin_matrix) if len(cl) > 0]
    if not unmix_tst:
        continue

    # From each cluster of TFs pick the one covering the most targets, then rank the picks by out/in ratio
    max_coverage_tfs = [str(tf) for tf in select_optimal_tfs(unmix_tst, state_out)]
    valid_tfs        = [tf for tf in max_coverage_tfs if tf in state_degrees.index]
    ranked           = state_degrees.loc[valid_tfs].sort_values('out/in', ascending=False)

    state_master_regulators_all[state_id] = ranked.index.tolist()
    print(f"State {state_id} (n={len(state_pts_in_eig)}): {len(ranked)} master regulators — {ranked.index[:5].tolist()}")

# Summary across states
all_masters = sorted(set(tf for tfs in state_master_regulators_all.values() for tf in tfs))
print(f"\nStates with master regulators: {len(state_master_regulators_all)} / {len(transcriptional_states)}")
print(f"Unique master regulators: {len(all_masters)}")
print(all_masters)

State 0 (n=38): 14 master regulators — ['FOXC2', 'MAZ', 'SP1', 'TCF3', 'KLF15']
State 1 (n=37): 15 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'KLF15', 'SP1']
State 2 (n=20): 17 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'NKX2-5', 'TCF3']
State 3 (n=20): 16 master regulators — ['MSX2', 'MAZ', 'TCF3', 'KLF15', 'SP1']
State 4 (n=18): 13 master regulators — ['MSX2', 'FOXC2', 'SP1', 'TCF3', 'KLF15']
State 5 (n=8): 14 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'TCF3', 'KLF15']
State 6 (n=53): 17 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'SP1', 'TCF3']
State 7 (n=21): 16 master regulators — ['FOXC2', 'MAZ', 'KLF15', 'TCF3', 'SP1']


State 8 (n=10): 17 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'TCF3', 'SP1']
State 9 (n=6): 16 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'SP1', 'TCF3']
State 10 (n=33): 14 master regulators — ['MSX2', 'MAZ', 'SP1', 'TCF3', 'HSF1']
State 11 (n=14): 13 master regulators — ['ZNF35', 'FOXC2', 'MAZ', 'TEAD3', 'SP1']
State 12 (n=13): 14 master regulators — ['MAZ', 'FOXC2', 'TEAD3', 'SP1', 'TCF3']
State 13 (n=31): 16 master regulators — ['ZNF35', 'MAZ', 'FOXC2', 'TEAD3', 'SP1']
State 14 (n=48): 13 master regulators — ['FOXC2', 'MAZ', 'TCF3', 'SP1', 'TEAD3']
State 15 (n=18): 16 master regulators — ['FOXC2', 'NKX2-5', 'SP1', 'TCF3', 'KLF15']


State 16 (n=28): 14 master regulators — ['MSX2', 'FOXC2', 'TCF3', 'SP1', 'KLF15']
State 17 (n=25): 14 master regulators — ['MSX2', 'FOXC2', 'MAZ', 'TEAD3', 'SP1']
State 18 (n=15): 17 master regulators — ['MSX2', 'FOXC2', 'TCF3', 'SP1', 'KLF15']


State 19 (n=47): 14 master regulators — ['FOXC2', 'MAZ', 'KLF15', 'TCF3', 'SP1']
State 20 (n=11): 11 master regulators — ['MSX2', 'MAZ', 'KLF11', 'NKX2-5', 'SP1']
State 21 (n=12): 16 master regulators — ['MSX2', 'FOXC2', 'TCF3', 'SP1', 'NR3C1']
State 22 (n=15): 14 master regulators — ['ZNF263', 'MSX2', 'FOXC2', 'MAZ', 'TCF3']
State 23 (n=9): 11 master regulators — ['MSX2', 'TFAP2A', 'SP1', 'EGR1', 'KLF15']

States with master regulators: 24 / 24
Unique master regulators: 74
['CDC5L', 'CNOT3', 'CREB3L2', 'DEAF1', 'DLX2', 'DLX6', 'E2F4', 'EGR1', 'ELF1', 'ELF2', 'ELK3', 'ESR2', 'ETV4', 'FOXC1', 'FOXC2', 'FOXD1', 'FOXF1', 'FOXF2', 'FOXO4', 'GBX2', 'HLTF', 'HMGA1', 'HOXA2', 'HOXA5', 'HOXA7', 'HOXA9', 'HOXD3', 'HOXD9', 'HSF1', 'HSF4', 'IRF5', 'KLF11', 'KLF15', 'LHX2', 'MAFG', 'MAZ', 'MEIS1', 'MEOX2', 'MLXIPL', 'MSX2', 'MYC', 'MYCN', 'NEUROD1', 'NF1', 'NFIB', 'NFKB2', 'NFYC', 'NKX2-5', 'NR2C2', 'NR3C1', 'NR4A2', 'NR6A1', 'PITX1', 'SIX6', 'SP1', 'SREBF1', 'STAT6', 'TCF3', 'TCF7L1', 'TEAD3', 'T

In [8]:
# Save master regulators as a long table: one row per (state, TF)
master_reg_df = pd.DataFrame([
    {'State': state_id, 'MasterRegulator': tf}
    for state_id, tfs in state_master_regulators_all.items() for tf in tfs
])
master_reg_df.to_csv(os.path.join(output_dir, "master_regulators_per_state.csv"), index=False)
print(f"Saved {len(master_reg_df)} master regulator-state associations")
print(f"Unique master regulators: {master_reg_df['MasterRegulator'].nunique()} | states covered: {master_reg_df['State'].nunique()}")

Saved 352 master regulator-state associations
Unique master regulators: 74 | states covered: 24
